In [1]:
# Remove packages that are not needed for this workshop
%pip uninstall -y diffusers gradio gradio-client

# Install compatible libraries
%pip install -q -U \
    "transformers>=4.51,<5" \
    "huggingface-hub>=0.34,<1" \
    accelerate \
    sentence-transformers \
    faiss-cpu \
    ipywidgets



Found existing installation: gradio_client 2.6.0
Uninstalling gradio_client-2.6.0:
  Successfully uninstalled gradio_client-2.6.0
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import pandas as pd
import torch
import faiss
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("FAISS:", faiss.__version__)

PyTorch: 2.14.0
GPU available: False
FAISS: 1.15.1


In [2]:
EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 9b0a0572-b1e5-4ae6-a828-953b1490a760)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 8b04a2cd-4f57-45df-ae5e-b9e93029baa8)')' thrown while requesting HEAD https:

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Dimension ->  384


/var/folders/lv/p9vlph914d5554g_z5tcrsdm0000gn/T/ipykernel_91004/1484055190.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())


In [3]:
sentences = ["This is an example sentence", "Each sentence is converted"]

embeddings = embedder.encode(
    sentences,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Number of Sentences -> ", len(sentences))
print("Embeddings Shape -> ", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of Sentences ->  2
Embeddings Shape ->  torch.Size([2, 384])


In [4]:
# Sentence A: I love playing cricket.
# Sentence B: Cricket is my favourite sport.
# Sentence C: The weather is very hot today.
# Compute Euclidian Distance, Dot Product and Cosine Similarity

sentences = ["I love playing cricket", "Cricket is my favourite sport", "The weather is very hot today"]

embeddings = embedder.encode(
    sentences,
    convert_to_numpy=True,
    show_progress_bar=True
)


dot_product = np.dot(embeddings[2], embeddings[1])
euclidian_distance = np.linalg.norm(embeddings[2] - embeddings[1])
cosine_similarity = np.dot(embeddings[2], embeddings[1]) / (np.linalg.norm(embeddings[2]) * np.linalg.norm(embeddings[1]))

print("Dot Product -> ", dot_product)
print("Euclidian Distance -> ", euclidian_distance)
print("Cosine Similarity -> ", cosine_similarity)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dot Product ->  0.12007312
Euclidian Distance ->  1.3265948
Cosine Similarity ->  0.12007312


In [5]:
documents = [
    "The university library closes at 10 PM.",
    "Students must maintain 75 percent attendance.",
    "The examination registration deadline is November 15.",
    "The hostel provides vegetarian and non-vegetarian meals.",
    "Scholarships are available for eligible students.",
    "The cricket team won the inter-college tournament.",
    "The computer science department offers AI courses.",
    "The university provides free Wi-Fi in the library.",
    "Students can borrow five books at a time.",
    "Hostel residents must return before 11 PM."
]

documnet_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings Shape -> ", documnet_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape ->  (10, 384)


In [6]:
def cosine_similarity(x,y):

  numerator = np.dot(x,y)
  denominator = np.linalg.norm(x) * np.linalg.norm(y)

  if denominator == 0:
    return 0.0

  return numerator / denominator

sim_1_2 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[1])

sim_1_3 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[2])

print("Similarity between Document 1 and Document 2 -> ", sim_1_2)
print("Similarity between Document 1 and Document 3 -> ", sim_1_3)


Similarity between Document 1 and Document 2 ->  0.15265742
Similarity between Document 1 and Document 3 ->  0.32226226
